# Stickler as a Strands Evals evaluator

Strands Evals scores structured output with `Equals`: whole-object `==`, so 0.0 or 1.0. Stickler scores
the same output field by field, so the number says *how* wrong it is and *which* field to fix.

Offline and deterministic. No credentials, no model calls, so the extraction is stubbed. For the same
evaluator with a live agent, see `Strands_Evals_FCC_Live_Agent.ipynb`.

## Install

This demo needs stickler plus the Strands Evals evaluator. The `StructuredOutput`
evaluator is in review at [sromoam/evals#1](https://github.com/sromoam/evals/pull/1) and is
not on PyPI yet, so install it from the branch:

```bash
pip install stickler-eval
pip install "strands-agents-evals @ git+https://github.com/sromoam/evals@feat/structured-output-evaluator"
```

Once that PR releases upstream, `pip install "strands-agents-evals[stickler]"` is all you need.

## Setup

In [1]:
import datetime
from typing import List, Optional

from pydantic import BaseModel, Field
from strands_evals import Case, Experiment, eval_task
from strands_evals.evaluators import Equals

from strands_evals.evaluators import StructuredOutput

## The model

A plain Pydantic model, the kind an agent already uses for `structured_output_model`. No stickler
annotations. Nullable fields are `Optional` because on a real invoice they are legitimately absent,
and "correctly returned nothing" has to score as a success.

In [2]:
class LineItem(BaseModel):
    sku: Optional[str] = None
    description: Optional[str] = None
    unit_price: Optional[float] = None


class Invoice(BaseModel):
    invoice_id: str
    vendor_name: str
    invoice_date: Optional[datetime.date] = None
    total_amount: Optional[float] = None
    line_items: List[LineItem] = Field(default_factory=list)

## Cases

Six invoices, each broken a different way: a missed field, an invented line item, several wrong
values, and two that should score perfectly. Between them they exercise the four categories that
occur when every labelled field carries a value — TP, FN, FA and FD. The fifth, TN, needs a field
absent on *both* sides; the sparse schema further down is where that one shows up.

`expected_output` is the label. `metadata` carries the stubbed extraction that the task returns.


In [3]:
def inv(iid, vendor, date, total, items):
    return Invoice(invoice_id=iid, vendor_name=vendor, invoice_date=date,
                   total_amount=total, line_items=[LineItem(**i) for i in items])


CASES = [
    # (name, ground truth, prediction)
    ("perfect",
     inv("INV-001", "Acme Corporation", "2026-01-15", 150.00,
         [{"sku": "SKU-1", "description": "Widget", "unit_price": 150.00}]),
     inv("INV-001", "Acme Corporation", "2026-01-15", 150.00,
         [{"sku": "SKU-1", "description": "Widget", "unit_price": 150.00}])),
    ("case-only",                                   # vendor differs only in case
     inv("INV-002", "Beta Industries", "2026-02-01", 220.00,
         [{"sku": "SKU-2", "description": "Gadget", "unit_price": 220.00}]),
     inv("INV-002", "BETA INDUSTRIES", "2026-02-01", 220.00,
         [{"sku": "SKU-2", "description": "Gadget", "unit_price": 220.00}])),
    ("amount-off",                                  # total slightly wrong
     inv("INV-003", "Gamma Ltd", "2026-02-20", 310.00,
         [{"sku": "SKU-3", "description": "Sprocket", "unit_price": 310.00}]),
     inv("INV-003", "Gamma Ltd", "2026-02-20", 311.50,
         [{"sku": "SKU-3", "description": "Sprocket", "unit_price": 310.00}])),
    ("missing",                                     # date not extracted -> FN
     inv("INV-004", "Delta LLC", "2026-03-05", 90.00,
         [{"sku": "SKU-4", "description": "Cog", "unit_price": 90.00}]),
     inv("INV-004", "Delta LLC", None, 90.00,
         [{"sku": "SKU-4", "description": "Cog", "unit_price": 90.00}])),
    ("extra-line",                                  # hallucinated row -> FA
     inv("INV-005", "Epsilon SA", "2026-03-19", 400.00,
         [{"sku": "SKU-5", "description": "Flange", "unit_price": 400.00}]),
     inv("INV-005", "Epsilon SA", "2026-03-19", 400.00,
         [{"sku": "SKU-5", "description": "Flange", "unit_price": 400.00},
          {"sku": "SKU-9", "description": "Phantom", "unit_price": 12.00}])),
    ("wrong",                                       # wrong on nearly everything
     inv("INV-006", "Zeta Holdings", "2026-04-02", 75.00,
         [{"sku": "SKU-6", "description": "Bracket", "unit_price": 75.00}]),
     inv("INV-999", "Omega Group", "2025-11-11", 12.00,
         [{"sku": "SKU-X", "description": "Unrelated", "unit_price": 12.00}])),
]

cases = [
    Case[str, Invoice](
        name=name,
        input=f"(OCR text for {name})",
        expected_output=ground_truth,
        metadata={"category": "invoice", "stub": prediction},
    )
    for name, ground_truth, prediction in CASES
]

print(f"{len(cases)} cases")

6 cases


## The task

`@eval_task()` wraps the function the harness calls per case. A real task would build an `Agent` here
and return its structured output; this one replays a stub so the notebook stays offline and both
evaluators see identical predictions.

Returning a `dict` matters: `EvalTaskHandler` passes a dict through untouched but calls `str()` on
anything else, which would flatten a Pydantic model into text.

In [4]:
@eval_task()
def extract(case):
    return {"output": case.metadata["stub"]}

## Equals versus stickler

Same cases, same harness, same task. Only the evaluator differs.

In [5]:
evaluator = StructuredOutput(Invoice)

stickler_report = await Experiment[str, Invoice](
    cases=cases, evaluators=[evaluator]
).run_evaluations_async(extract)

equals_report = await Experiment[str, Invoice](
    cases=cases, evaluators=[Equals()]
).run_evaluations_async(extract)

print(f"{'case':12} {'stickler':>9} {'equals':>7}")
print("-" * 30)
for c, s, e in zip(stickler_report.cases, stickler_report.scores, equals_report.scores):
    print(f"{c['name']:12} {s:>9.3f} {e:>7.1f}")

print(f"\noverall      {stickler_report.overall_score:>9.3f} {equals_report.overall_score:>7.3f}")
print(f"distinct     {len(set(round(v, 4) for v in stickler_report.scores)):>9} "
      f"{len(set(round(v, 4) for v in equals_report.scores)):>7}")

case          stickler  equals
------------------------------
perfect          1.000     1.0
case-only        1.000     0.0
amount-off       0.800     0.0
missing          0.800     0.0
extra-line       0.900     0.0
wrong            0.025     0.0

overall          0.754   0.167
distinct             4       2


`Equals` collapses five of six to 0.0, including the one that differed only in capitalisation, so it
cannot rank extractors or spot a regression. Stickler separates "one field slightly off" from "wrong
on everything".

The report renders itself too. Use `display()` in a notebook; `run_display()` is the interactive
variant and blocks waiting for keyboard input.

In [6]:
stickler_report.display(include_input=False)

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.75           Pass Rate: 0.8333333333333334                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                          Test Case Results                           
┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ index ┃ name       ┃ evaluator        ┃ score ┃ test_pass ┃ reason ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ ▶ 0   │ perfect    │ StructuredOutput │ 1.00  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 1   │ case-only  │ StructuredOutput │ 1.00  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 2   │ amount-off │ StructuredOutput │ 0.80  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 3   │ missing    │ StructuredOutput │ 0.80  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 4   │ extra-line │ StructuredOutput │ 0.90  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 5   │ wrong      │ StructuredOutput │ 0.03  │ ❌        │ ...    │
└───────┴────────────┴──────────────────┴───────┴───────────┴────────┘

## Per-case detail

`evaluate()` returns one `EvaluationOutput` per case carrying the weighted score. Field-level detail
comes from `per_case()`, read from the comparison the evaluator already performed. `EvaluationOutput`
has four scalar fields, so it could not carry this anyway.

In [7]:
# per_case() accumulates in completion order, and the harness runs cases
# concurrently, so sort back into case order to keep this cell reproducible.
ORDER = [c.name for c in cases]

for entry in sorted(evaluator.per_case(), key=lambda e: ORDER.index(e["case"])):
    weak = {f: round(s, 2) for f, s in entry["field_scores"].items() if s < 1.0}
    print(f"{entry['case']:12} {entry['overall_score']:>5.2f}  "
          f"pass={str(entry['test_pass']):5}  weak={weak or '-'}")

perfect       1.00  pass=True   weak=-
case-only     1.00  pass=True   weak=-
amount-off    0.80  pass=True   weak={'total_amount': 0.0}
missing       0.80  pass=True   weak={'invoice_date': 0.0}
extra-line    0.90  pass=True   weak={'line_items': 0.5}
wrong         0.03  pass=False  weak={'invoice_id': 0.0, 'vendor_name': 0.0, 'invoice_date': 0.0, 'total_amount': 0.0, 'line_items': 0.12}


## What `test_pass` does and does not promise

`test_pass` is a **document** verdict, not a per-field one, and it is worth knowing exactly what it
gates on before you build a CI check around it.

Two facts combine. First, fields are weighted uniformly by default, so on a five-field invoice one
entirely wrong field costs only 0.2 and the document still clears the 0.7 default. Look at
`amount-off` above: it has `total_amount: 0.0` and a passing score. Second, a field absent on
**both** sides scores 1.0, on the reasoning that a value the model correctly left blank is a value it
got right. On a sparse schema those uninformative fields outvote the informative ones.

Taken to its conclusion, the second fact means a prediction that found *nothing* can out-score a
schema's populated fields. The evaluator therefore requires **both** the weighted score and `recall`
to clear `match_threshold`. Recall counts only fields that had a value to find, so correct absence
neither helps nor hurts it. The cell below shows the case that motivates the gate.


In [8]:
# A sparse schema: two fields that matter, a long optional tail. The common shape.
class SparseInvoice(BaseModel):
    invoice_id: str
    vendor_name: str
    invoice_date: Optional[str] = None
    total_amount: Optional[float] = None
    tax_amount: Optional[float] = None
    po_number: Optional[str] = None
    payment_terms: Optional[str] = None
    shipping_address: Optional[str] = None
    billing_address: Optional[str] = None
    notes: Optional[str] = None


truth = SparseInvoice(invoice_id='INV-8842', vendor_name='Acme Corporation')

SCENARIOS = {
    'found nothing': SparseInvoice(invoice_id='', vendor_name=''),
    'both correct': SparseInvoice(invoice_id='INV-8842', vendor_name='Acme Corporation'),
    'vendor wrong': SparseInvoice(invoice_id='INV-8842', vendor_name='Zenith Industries'),
}

sparse_cases = [
    Case[str, SparseInvoice](name=label, input='(scan)', expected_output=truth,
                             metadata={'stub': pred})
    for label, pred in SCENARIOS.items()
]

sparse_eval = StructuredOutput(SparseInvoice)
await Experiment[str, SparseInvoice](
    cases=sparse_cases, evaluators=[sparse_eval]
).run_evaluations_async(extract)

print(f"{'scenario':16} {'score':>6} {'recall':>7} {'test_pass':>10}")
print('-' * 43)
for entry in sorted(sparse_eval.per_case(), key=lambda e: list(SCENARIOS).index(e['case'])):
    rec = entry['recall']
    rec_s = f"{rec:.3f}" if rec is not None else 'n/a'
    print(f"{entry['case']:16} {entry['overall_score']:>6.3f} {rec_s:>7} "
          f"{str(entry['test_pass']):>10}")

print('\n`found nothing` scores 0.8 because eight fields are blank on both sides,')
print('but recall is 0.0, so it does not pass. That is the gate doing its job.')


scenario          score  recall  test_pass
-------------------------------------------
found nothing     0.800   0.000      False
both correct      1.000   1.000       True
vendor wrong      0.900   1.000       True

`found nothing` scores 0.8 because eight fields are blank on both sides,
but recall is 0.0, so it does not pass. That is the gate doing its job.


If you want a stricter check than the default, three options in increasing strictness: raise
`match_threshold`, gate on named fields from `per_case()['field_scores']`, or read `fd` from
`metrics()`, which counts every wrong field whatever the document verdict said. Raising `weight` on
the fields you expect to be populated is often the cleanest fix, because it makes the score itself
reflect what you care about. See
[Sparse Objects](https://awslabs.github.io/stickler/Getting-Started/thresholds-and-metrics/#sparse-objects).


## Dataset rollup

`metrics()` returns stickler's five-category confusion matrix per field path, including nested paths.
It runs no extra comparisons: each case was compared once above and the raw result was kept.

**FN** is a field the extractor missed, **FA** one it invented, **FD** one it got wrong. A single
score cannot separate those, and they need different fixes.

In [9]:
rollup = evaluator.metrics()["Invoice"]

print(f"documents: {rollup.document_count}\n")
print(f"{'field':26} {'tp':>3} {'fn':>3} {'fa':>3} {'fd':>3}  {'prec':>5} {'rec':>5} {'f1':>5}")
print("-" * 62)
for path, m in sorted(rollup.field_metrics.items(),
                      key=lambda kv: (kv[1].get("cm_f1", 1.0), kv[0])):
    print(f"{path:26} {m.get('tp', 0):>3} {m.get('fn', 0):>3} {m.get('fa', 0):>3} {m.get('fd', 0):>3}"
          f"  {m.get('cm_precision', 0):>5.2f} {m.get('cm_recall', 0):>5.2f} {m.get('cm_f1', 0):>5.2f}")

documents: 6

field                       tp  fn  fa  fd   prec   rec    f1
--------------------------------------------------------------
total_amount                 4   0   0   2   0.67  1.00  0.80
invoice_date                 4   1   0   1   0.80  0.80  0.80
line_items                   5   0   1   1   0.71  1.00  0.83
invoice_id                   5   0   0   1   0.83  1.00  0.91
vendor_name                  5   0   0   1   0.83  1.00  0.91
line_items.description       5   0   0   0   1.00  1.00  1.00
line_items.sku               5   0   0   0   1.00  1.00  1.00
line_items.unit_price        5   0   0   0   1.00  1.00  1.00


The table is sorted by ascending F1, so the top row is the weakest field — the practical entry point
when you have a regression and no idea where it is.

Across these 6 documents the five categories separate three different problems:

- **`total_amount` reads `fd=2`, `fn=0`.** Never missed, wrong twice. That is a parsing problem, not a
  detection one; a metric that reported both as "failure" would have pointed at the wrong fix.
- **`invoice_date` is the only row with `fn=1`.** One document had a date the extractor did not return at
  all, which is why it is the only field whose recall is below `1.00`.
- **`line_items` is the only row with `fa=1`.** One predicted element matched nothing in the label — a
  hallucinated line item. No other field invented a value.

Now read the nested rows carefully, because they do not mean what the top-level rows mean. Two
things are different about them.

First, they are counted **per matched line-item pair, not per document**. Every label in this dataset
carries exactly one line item, so a pair count and a document count happen to coincide here; on a
dataset with several items per invoice the child rows would total well above `document_count`, and
reading them as a per-document rate would understate the error rate badly.

Second, a pair only contributes child rows if it scored at or above `match_threshold`. Below that,
threshold gating treats the pair as atomic and emits no field breakdown, so the document appears as `fd`
on `line_items` and is absent from the child rows entirely. Both effects are visible in the same number:
every child row reads `tp=5` against 6 documents because one document's pair was gated out — the same
document that supplies `line_items`' `fd=1`. So `line_items.sku` at `f1=1.00` means the SKU was right on
all five pairs close enough to look inside, **not** that it was right five times out of six.

That gating is also why the row set is data-dependent rather than fixed by the schema: had no pair
cleared the threshold, there would be no `line_items.*` rows at all. Use `.get()` rather than indexing.
Nested leaves carry counts and precision/recall/F1 but no mean score
([#249](https://github.com/awslabs/stickler/issues/249)).


## Why each field scored that way

Nothing was configured, so every comparator and threshold was inferred from the model. `explain()`
shows what was chosen and on what basis, which is what makes a score defensible.

In [10]:
print(f"{'field':26} {'comparator':24} {'thr':>5}  basis")
print("-" * 68)
for path, cfg in evaluator.explain().items():
    print(f"{path:26} {cfg['comparator']:24} {cfg['threshold']:>5}  {cfg['source']}")

field                      comparator                 thr  basis
--------------------------------------------------------------------
invoice_id                 ExactComparator            1.0  name-token
vendor_name                LevenshteinComparator     0.85  name-token
invoice_date               DateComparator            0.95  name-token
total_amount               NumericComparator         0.95  name-token
line_items                 Hungarian (per-element StructuredModel)   0.7  type
line_items.sku             ExactComparator            1.0  name-token
line_items.description     FuzzyComparator            0.6  name-token
line_items.unit_price      NumericComparator         0.95  name-token


## Mixed output types

`model_cls` is optional. Omit it and the model class is inferred per case, and `metrics()` partitions
its rollup by class. This matters because feeding two schemas into one rollup would union their field
paths, making a field present in half the documents look missed in the rest.

In [11]:
class Receipt(BaseModel):
    merchant: str
    tax: float


mixed = [
    Case[str, Invoice](name="inv-1", input="", expected_output=CASES[0][1],
                       metadata={"stub": CASES[0][2]}),
    Case[str, Receipt](name="rec-1", input="", expected_output=Receipt(merchant="Corner Store", tax=4.50),
                       metadata={"stub": Receipt(merchant="Corner Store", tax=9.99)}),
]

mixed_eval = StructuredOutput()          # no model_cls
await Experiment(cases=mixed, evaluators=[mixed_eval]).run_evaluations_async(extract)

for model_name, pe in mixed_eval.metrics().items():
    print(f"{model_name:10} docs={pe.document_count}  fields={sorted(pe.field_metrics)}")

Invoice    docs=1  fields=['invoice_date', 'invoice_id', 'line_items', 'line_items.description', 'line_items.sku', 'line_items.unit_price', 'total_amount', 'vendor_name']
Receipt    docs=1  fields=['merchant', 'tax']


Pass `model_cls` instead and the evaluator is strict: anything that will not validate as that class
raises, which is what a single-schema suite wants.

## Notes for reuse

The evaluator accumulates across cases, so call `reset()` before reusing an instance for a second run.

It is safe at any concurrency. During a run it only appends to a list, which is atomic under the GIL,
so it holds no locks and there is no need to pin `max_workers`. All aggregation happens afterwards.

In [12]:
print(f"before reset: {rollup.document_count} documents")
evaluator.reset()
print(f"after reset:  {evaluator.metrics()}  {evaluator.per_case()}")

# metrics() rebuilds on every call, so a rollup you already hold is a snapshot.
print(f"rollup held across the reset: {rollup.document_count} documents")


before reset: 6 documents
after reset:  {}  []
rollup held across the reset: 6 documents


`reset()` clears both surfaces: `metrics()` returns an empty dict and `per_case()` an empty list, so the
next run starts from nothing. Without it a second run's numbers are silently averaged with the first,
which looks like a modest regression rather than a bug in your harness.

Note that `rollup` still reports 6 documents afterwards. `metrics()` builds a fresh object each call, so
a rollup you already hold is a snapshot, not a live view — which makes it safe to keep across a reset if
you want to compare two runs.
